
# 02 — Rotary Position Embeddings (RoPE), from Scratch

**Goal:** implement RoPE, prove to yourself (numerically) that it makes attention scores a
function of *relative* position only, and be fluent on why it's applied to Q/K but not V, why
`base=10000`, and what happens (and what's done about it) when you generate past the context
length the model was trained on. This is one of the most commonly cited "implement this"
questions in current LLM interviews — nearly every modern open-weight LLM (Llama, Qwen,
Mistral, GPT-NeoX, ...) uses it.

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 The problem RoPE solves

Transformers have no inherent notion of token order — attention is a set operation over
positions unless you inject position information somewhere. The original Transformer added a
fixed sinusoidal (or learned) positional vector directly to the token embedding before the first
layer. That approach has two weaknesses interviewers like to probe:

1. It encodes **absolute** position into the residual stream, so the model has to *learn* to
   recover relative relationships (like "token j is 3 positions before token i") indirectly —
   the thing attention actually wants to reason about is often relative distance, not absolute
   index.
2. It doesn't naturally extrapolate: a sinusoidal table built for positions $0..L_{max}-1$ says
   nothing well-defined about position $L_{max}$, and learned absolute embeddings have *no*
   representation at all for positions beyond what was trained.

RoPE takes a different approach: instead of adding a position vector to the embedding, it
**rotates** the query and key vectors by an angle that depends on their position, at attention
time, per head, per pair of dimensions. The rotation is designed so that the dot product between
a rotated query at position $i$ and a rotated key at position $j$ depends *only* on $i - j$ — the
model gets relative-position-aware attention scores "for free," directly from the geometry, with
no absolute-position vector ever added to the residual stream.

### 1.2 The construction

Split the head dimension $d$ into $d/2$ pairs. For pair $m$ (using 0-indexing over pairs), define
a frequency
$$
\theta_m = \text{base}^{-2m/d}, \qquad m = 0, \dots, d/2-1
$$
At sequence position $t$, rotate the $m$-th pair of the vector (whatever "pair" means under your
chosen convention — see the note in the implementation below) by angle $t\theta_m$:
$$
\begin{pmatrix}x'_a \\ x'_b\end{pmatrix} = \begin{pmatrix}\cos(t\theta_m) & -\sin(t\theta_m)\\ \sin(t\theta_m) & \cos(t\theta_m)\end{pmatrix}\begin{pmatrix}x_a \\ x_b\end{pmatrix}
$$
Equivalently, treating each pair $(x_a, x_b)$ as a complex number $x_a + i x_b$, this rotation is
just multiplication by $e^{i t\theta_m}$. This equivalence is worth having in your back pocket —
it makes the "why does the dot product only depend on $i-j$" argument a one-liner: rotating $q$
by $e^{i\cdot i\theta_m}$ and $k$ by $e^{i\cdot j\theta_m}$, then taking $q \cdot \bar{k}$
(complex conjugate dot product, which recovers the real 2D dot product), multiplies the phases
as $e^{i(i-j)\theta_m}$ — the absolute positions $i,j$ only ever appear through their
*difference*.

### 1.3 Which pairs? Two conventions you'll see

- **Interleaved** (the original RoPE paper's figure): pair up adjacent dimensions,
  $(x_0,x_1), (x_2,x_3), \dots$
- **Split-half / "rotate_half"** (what GPT-NeoX, Llama, and most production implementations
  actually use, because it's a cheaper set of tensor ops): pair dimension $m$ with dimension
  $m + d/2$, i.e. $(x_0, x_{d/2}), (x_1, x_{d/2+1}), \dots$, all $d/2$ pairs computed at once via
  a "rotate half the vector, combine with cos/sin tables" trick instead of an explicit
  per-pair loop.

Both are valid RoPE — they're the same rotation, just applied to a different assignment of which
coordinates count as "a pair." We implement the split-half/rotate_half convention below since
that's what you'll actually encounter reading real model code.

### 1.4 Why Q and K, never V

The entire point of RoRE is to make the **attention score** $Q_i \cdot K_j$ a function of
relative position. $V$ is not involved in computing attention scores at all — it's the content
that gets mixed together *after* attention weights (which already encode relative position via
Q/K) are computed. Rotating $V$ would apply an arbitrary transformation to content with no
mechanism for anything downstream to make sense of it — there's nothing analogous to the
"rotate query, rotate key, take dot product, phases cancel to a relative offset" argument for a
value vector that's just going to be summed with attention weights.

### 1.5 Context-length extrapolation

Because $\theta_m$ and the rotation angles $t\theta_m$ are well-defined for *any* integer $t$
(not just $t < L_{train}$), RoPE in principle "extrapolates" better than a fixed lookup-table
positional embedding. In practice, naive extrapolation still degrades because the model never
saw attention patterns at those large relative offsets during training — the fix used in
practice is some form of **position interpolation** (compress positions beyond the trained
range back into the trained range, e.g. by scaling $t$) or **NTK-aware scaling** (rescale the
base frequencies non-uniformly, stretching low frequencies more than high frequencies, which
preserves fine-grained relative structure at short distances while extending the range at long
distances). Interviewers usually just want you to know RoPE doesn't magically solve length
extrapolation for free, and that these are the standard mitigations.

### 1.6 Contrast with ALiBi

An alternative approach (ALiBi) skips rotating Q/K entirely and instead adds a fixed,
non-learned *penalty* to attention scores proportional to $-(i-j)$ (a linear bias favoring
nearby tokens), directly in score space rather than through a geometric rotation. Both RoPE and
ALiBi target the same goal (relative-position-aware attention with good extrapolation) via very
different mechanisms — RoPE via rotation before the dot product, ALiBi via an additive bias
after it. Knowing this contrast is a common quick follow-up.


In [ ]:

import torch
import torch.nn as nn
import math

torch.manual_seed(0)



## 2. Implementation

Implement RoPE using the split-half / `rotate_half` convention. Fill in every `# TODO`.

- `build_rope_cache(seq_len, head_dim, base=10000.0)` → returns `(cos, sin)`, each of shape
  `(seq_len, head_dim)` (the frequency table is computed for `head_dim/2` pairs, then
  duplicated across both halves so `cos`/`sin` broadcast cleanly against a full `head_dim`
  vector in `apply_rope`).
- `rotate_half(x)` → for `x` of shape `(..., head_dim)`, split into two halves along the last
  dim, return `cat([-second_half, first_half])`.
- `apply_rope(x, cos, sin)` → `x * cos + rotate_half(x) * sin`, where `x` is `(..., seq_len,
  head_dim)` and `cos`/`sin` are `(seq_len, head_dim)` (broadcast over any leading dims).


In [ ]:

def build_rope_cache(seq_len: int, head_dim: int, base: float = 10000.0):
    assert head_dim % 2 == 0, "head_dim must be even to form pairs"
    # TODO: inv_freq = 1 / base^(2m/head_dim) for m = 0 .. head_dim/2 - 1, shape (head_dim/2,)
    inv_freq = None

    # TODO: t = 0 .. seq_len-1, shape (seq_len,)
    t = None

    # TODO: freqs = outer product of t and inv_freq -> (seq_len, head_dim/2)
    freqs = None

    # TODO: duplicate freqs across both halves -> (seq_len, head_dim), then return (cos, sin)
    cos = None
    sin = None
    return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    # TODO: split x's last dim in half, return cat([-second_half, first_half], dim=-1)
    raise NotImplementedError


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # TODO: x * cos + rotate_half(x) * sin  (cos/sin broadcast over x's leading dims)
    raise NotImplementedError



### Sanity tests

1. Shape check.
2. **The core claim**: the dot product between a rotated query at position $i$ and a rotated
   key at position $j$ depends only on $i - j$ — verified by computing it at several $(i,j)$
   pairs sharing the same offset and checking they're identical, then checking a different
   offset gives a different value.
3. **Cross-check against the complex-number formulation** — an independent implementation path
   that should agree with `apply_rope` to numerical precision if your rotation is correct.


In [ ]:

seq_len, head_dim = 10, 8
cos, sin = build_rope_cache(seq_len, head_dim)
assert cos.shape == (seq_len, head_dim) and sin.shape == (seq_len, head_dim)

q_batch = torch.randn(2, 4, seq_len, head_dim)  # (batch, heads, seq, head_dim)
q_rot = apply_rope(q_batch, cos, sin)
assert q_rot.shape == q_batch.shape
print("Test 1 passed: shapes correct")


In [ ]:

torch.manual_seed(1)
q_vec = torch.randn(head_dim)
k_vec = torch.randn(head_dim)

def rotated_dot(i, j):
    qi = apply_rope(q_vec, cos[i], sin[i])
    kj = apply_rope(k_vec, cos[j], sin[j])
    return (qi * kj).sum().item()

offset = 2
vals = [rotated_dot(i, i - offset) for i in range(offset, seq_len)]
for v in vals[1:]:
    assert abs(v - vals[0]) < 1e-4, f"same-offset dot products should match: {v} vs {vals[0]}"

val_other_offset = rotated_dot(5, 1)  # offset 4, should differ from offset-2 dots
assert abs(val_other_offset - vals[0]) > 1e-3

print(f"Test 2 passed: dot product is constant ({vals[0]:.4f}) across all pairs with offset={offset}, "
      f"and differs ({val_other_offset:.4f}) at a different offset")


In [ ]:

# Independent cross-check via the complex-number formulation. NOTE: this must use the SAME
# pairing convention as apply_rope (split-half: pair m is (x[m], x[m+head_dim/2])), not
# interleaved pairing -- they are both valid RoPE conventions but are not interchangeable
# within a single check.
def complex_rope_dot(q_vec, k_vec, i, j, base=10000.0):
    d = head_dim
    half = d // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    q_complex = torch.complex(q_vec[:half], q_vec[half:])
    k_complex = torch.complex(k_vec[:half], k_vec[half:])
    q_rot = q_complex * torch.exp(1j * inv_freq * i)
    k_rot = k_complex * torch.exp(1j * inv_freq * j)
    return (q_rot * k_rot.conj()).real.sum().item()

i, j = 7, 3
val_matrix_form = rotated_dot(i, j)
val_complex_form = complex_rope_dot(q_vec, k_vec, i, j)
assert abs(val_matrix_form - val_complex_form) < 1e-3, (val_matrix_form, val_complex_form)
print(f"Test 3 passed: rotate_half form ({val_matrix_form:.4f}) matches complex-number form "
      f"({val_complex_form:.4f})")



## 3. Quiz

1. Why rotate the vector in 2D *pairs* rather than applying some single rotation to the whole
   $d$-dimensional vector at once?
2. Why `base=10000` specifically — what would changing it (smaller or larger) do to the
   frequency spectrum across pairs, and why does the spectrum need a spread of frequencies at
   all rather than one fixed frequency?
3. Derive (in your own words, using the complex-number view) why $Q_i \cdot K_j$ after RoPE
   depends only on $i - j$.
4. Why is RoPE applied to Q and K but never to V?
5. What happens if you naively run a RoPE model on sequences longer than it was trained on, and
   what do position interpolation / NTK-aware scaling each do about it?
6. Contrast RoPE with ALiBi: both target relative-position-aware attention — where in the
   computation does each one inject that information, and what's a practical consequence of the
   difference (e.g., for extrapolation or for combining with other techniques like caching)?
7. In a real transformer layer with a KV cache (notebook 07), at which point do you apply RoPE
   to a newly-computed K vector — before or after inserting it into the cache — and why does
   that choice matter?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Why 2D pairs, not one big rotation
A single rotation of the full $d$-dimensional vector would only have *one* degree of freedom (one
angle), giving the model exactly one frequency at which to encode position — far too coarse to
capture both fine-grained (nearby-token) and coarse-grained (far-apart-token) relative
relationships. Splitting into $d/2$ independent 2D rotations, each with its *own* frequency
$\theta_m$, gives the model a whole spectrum of frequencies simultaneously: some pairs rotate
quickly (high frequency — sensitive to small changes in relative position, good for local
structure) and some rotate slowly (low frequency — stay coherent over long relative distances,
good for long-range structure). This mirrors exactly why the original sinusoidal position
embeddings used a spread of frequencies too — RoPE reuses that same "spectrum" idea, but applies
it as a rotation of Q/K rather than as an additive embedding.

### Q2 — Why base=10000
$\theta_m = \text{base}^{-2m/d}$ ranges from $\theta_0 = 1$ (fastest-rotating pair) down to a
very small value at $m = d/2-1$ (slowest-rotating pair), and `base` controls how spread out that
range of frequencies is. A larger base spreads the frequencies further apart (pushes the slowest
frequencies even slower, letting the model represent very long relative distances without the
slowest components "wrapping around"/aliasing), while a smaller base compresses the spectrum
into a narrower band. `base=10000` was an empirical choice in the original RoPE/Transformer
lineage that gives a good spread for typical training context lengths; it's exactly the kind of
knob that later extrapolation tricks (NTK-aware scaling) adjust when you want to serve
longer contexts than trained without retraining from scratch — you need *some* spread of
frequencies rather than a single one specifically so different pairs specialize in different
distance scales simultaneously; collapsing to one frequency loses that.

### Q3 — The complex-number derivation
Treat each rotated pair as a complex number. RoPE rotates $q$ at position $i$ to
$q \cdot e^{i\theta_m \cdot i}$ (per pair $m$) and $k$ at position $j$ to
$k \cdot e^{i\theta_m \cdot j}$. The real 2D dot product of two rotated pairs equals
$\text{Re}(q_{rot} \cdot \overline{k_{rot}})$ (real part of one times the complex conjugate of
the other). Substituting:
$$
\text{Re}\left(q\, e^{i\theta_m i} \cdot \overline{k\, e^{i\theta_m j}}\right)
= \text{Re}\left(q\bar{k}\, e^{i\theta_m i} e^{-i\theta_m j}\right)
= \text{Re}\left(q\bar{k}\, e^{i\theta_m (i-j)}\right)
$$
The absolute positions $i$ and $j$ only ever appear combined as $(i-j)$ inside the phase — no
matter what $i,j$ individually are, the result is identical for any pair with the same
difference. Summing this over all $d/2$ pairs (each with its own $\theta_m$, all still only
depending on $i-j$) gives the full attention score's relative-position-only dependence.

### Q4 — Why not V
Rotating Q and K serves a specific mechanical purpose: it makes the score $Q_i \cdot K_j$, which
is computed as a dot product, carry relative-position information through phase cancellation as
derived in Q3. $V$ never participates in a dot product that phases could cancel in — it's simply
the content vector that gets linearly combined according to attention weights *after* those
weights (already relative-position-aware, thanks to Q/K) are finalized. There is no analogous
"cancellation" mechanism for $V$, and nothing downstream ever un-rotates it, so rotating $V$
would just be an arbitrary, purposeless transformation of the content being mixed — it wouldn't
encode anything and would need to be inverted to preserve meaning, which nothing does.

### Q5 — Extrapolation
RoPE's rotation angles are mathematically defined for any position $t$, so nothing crashes when
generating past the trained length — but the model's attention behavior at relative offsets it
never saw during training tends to degrade (attention patterns the model learned were tuned to a
particular range of $t\theta_m$ values). **Position interpolation** addresses this by rescaling
positions so that a longer *actual* sequence still produces $t\theta_m$ values within the
originally-trained range (compressing far-apart tokens' effective relative distance back into
familiar territory), at some cost to fine-grained resolution. **NTK-aware scaling** instead
rescales the base/frequencies non-uniformly — stretching low frequencies (long-range components)
more aggressively while leaving high frequencies (short-range, fine local structure) closer to
their original values — aiming to extend usable range with less degradation to local attention
patterns than uniform interpolation causes. Both are post-hoc mitigations, not something RoPE
gives you automatically.

### Q6 — RoPE vs. ALiBi
RoPE injects relative-position information *before* the dot product, by rotating Q and K so that
their dot product's phase naturally encodes $i-j$. ALiBi instead leaves Q/K untouched and adds a
fixed linear penalty *after* the dot product, directly to the raw attention score:
$\text{score}_{ij} \mathrel{-}= m\cdot(i-j)$ for some per-head slope $m$, favoring attending to
nearby tokens. A practical consequence: ALiBi's bias is trivial to compute at any sequence length
(it's just an arithmetic penalty, no lookup table or rotation needed) and tends to extrapolate to
longer contexts more gracefully out-of-the-box, whereas RoPE's rotation-based encoding is more
expressive (a full geometric mechanism rather than one linear penalty) but needs the
interpolation/NTK-scaling tricks from Q5 to extrapolate well. This is a real design trade-off,
not a strictly-better-or-worse comparison — which is why some newer architectures use RoPE, some
use ALiBi, and some combine ideas from both.

### Q7 — RoPE and the KV cache
RoPE must be applied to a new token's $K$ vector **before** it's written into the cache (i.e.,
you rotate $K$ by its actual absolute position, then cache the *rotated* result) — because the
rotation angle depends on that token's position, which is fixed and known the moment it's
computed and never changes again. Caching the *unrotated* $K$ and rotating on every subsequent
read would be both wasteful (repeating the same rotation every step) and error-prone (you'd need
to track, for every cached position, what rotation to apply on each future read — pure
bookkeeping overhead for zero benefit, since the rotated value never changes once computed).
Applying RoPE once, at insertion time, and caching the rotated result directly is both simpler
and strictly cheaper.
